### 1. 구글 드라이브 마운트

In [1]:
# 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### 2. 각 파일 내의 엑셀 파일 로드

In [2]:
import os
import pandas as pd

# 기본 경로 설정
base_path = '/content/drive/MyDrive/26 국어정보학/기말과제/'

# 키워드별 디렉토리 정의
keyword_dirs = {
    '화제': os.path.join(base_path, 'CollectedData_2018-2025_화제'),
    '논란': os.path.join(base_path, 'CollectedData_2018-2025_논란'),
    '이슈': os.path.join(base_path, 'CollectedData_2018-2025_이슈')
}

# 로드된 모든 엑셀 파일을 저장할 딕셔너리
# 구조: {키워드: {파일이름: DataFrame}}
all_excel_files = {}

print("엑셀 파일 로딩 시작...")

for keyword, dir_path in keyword_dirs.items():
    if os.path.exists(dir_path):
        print(f"\n디렉토리 '{dir_path}'에서 파일 목록을 가져옵니다.")
        # 해당 키워드에 대한 딕셔너리 초기화
        all_excel_files[keyword] = {}

        # 디렉토리 내 파일 목록을 순회
        for filename in os.listdir(dir_path):
            # .xlsx 또는 .xls 확장자를 가진 파일만 처리
            if filename.endswith(('.xlsx', '.xls')):
                file_path = os.path.join(dir_path, filename)
                try:
                    df = pd.read_excel(file_path)
                    all_excel_files[keyword][filename] = df
                    print(f"  - '{filename}' (키워드: {keyword}) 로드 완료. ({df.shape[0]} 행, {df.shape[1]} 열)")
                except Exception as e:
                    print(f"  - 오류: '{filename}' (키워드: {keyword}) 로드 중 문제 발생: {e}")
    else:
        print(f"\n오류: 디렉토리 '{dir_path}'를 찾을 수 없습니다. 경로를 확인해주세요.")

print("\n--- 엑셀 파일 로딩 요약 ---")
if not all_excel_files:
    print("로드된 엑셀 파일이 없습니다. 경로와 파일 존재 여부를 확인해주세요.")
else:
    for keyword, files in all_excel_files.items():
        print(f"키워드: '{keyword}', 로드된 파일 수: {len(files)}개")
        for filename, df in files.items():
            print(f"  - 파일명: '{filename}', 데이터프레임 크기: {df.shape}")

print("\n모든 엑셀 파일 로딩 완료. 'all_excel_files' 변수에 저장되었습니다.")


엑셀 파일 로딩 시작...

디렉토리 '/content/drive/MyDrive/26 국어정보학/기말과제/CollectedData_2018-2025_화제'에서 파일 목록을 가져옵니다.
  - 'NewsResult_전국일간지_화제_20180101-20181231.xlsx' (키워드: 화제) 로드 완료. (2985 행, 8 열)
  - 'NewsResult_전국일간지_화제_20190101-20191231.xlsx' (키워드: 화제) 로드 완료. (2919 행, 8 열)
  - 'NewsResult_전국일간지_화제_20200101-20201231.xlsx' (키워드: 화제) 로드 완료. (2854 행, 8 열)
  - 'NewsResult_전국일간지_화제_20210101-20211231.xlsx' (키워드: 화제) 로드 완료. (2388 행, 8 열)
  - 'NewsResult_전국일간지_화제_20220101-20221231.xlsx' (키워드: 화제) 로드 완료. (1844 행, 8 열)
  - 'NewsResult_전국일간지_화제_20230101-20231231.xlsx' (키워드: 화제) 로드 완료. (2097 행, 8 열)
  - 'NewsResult_전국일간지_화제_20240101-20241231.xlsx' (키워드: 화제) 로드 완료. (2174 행, 8 열)
  - 'NewsResult_전국일간지_화제_20250101-20251231.xlsx' (키워드: 화제) 로드 완료. (2162 행, 8 열)

디렉토리 '/content/drive/MyDrive/26 국어정보학/기말과제/CollectedData_2018-2025_논란'에서 파일 목록을 가져옵니다.
  - 'NewsResult_전국일간지_논란_20240101-20241231.xlsx' (키워드: 논란) 로드 완료. (1

### 3. 메타데이터 파일 생성 및 저장

In [ ]:
import re

# 메타데이터를 저장할 리스트
metadata_list = []

print("메타데이터 추출 시작...")

for keyword, files_dict in all_excel_files.items():
    for filename, df in files_dict.items():
        # 파일명에서 연도 추출 (예: NewsResult_..._20180101-20181231.xlsx 에서 2018 추출)
        year_match = re.search(r'_(\d{4})\d{4}-\d{8}\.xlsx', filename)
        year = int(year_match.group(1)) if year_match else 'N/A'

        # 언론사별 개수 계산
        media_counts = {}
        if '언론사' in df.columns:
            media_counts = df['언론사'].value_counts().to_dict()

        metadata_list.append({
            '키워드': keyword,
            '연도': year,
            '파일명': filename,
            '총_행_수': df.shape[0],
            '총_열_수': df.shape[1],
            '컬럼_이름': list(df.columns),
            '언론사_별_개수': media_counts # 새로 추가된 부분
        })

# 메타데이터 리스트를 DataFrame으로 변환
metadata_df = pd.DataFrame(metadata_list)

print("\n--- 메타데이터 정리 완료 ---")
print(f"총 {len(metadata_df)}개의 파일에 대한 메타데이터가 정리되었습니다.")
print("\n메타데이터 DataFrame 미리보기:")
display(metadata_df.head())

print("\n키워드별, 연도별 데이터 요약:")
# 키워드별, 연도별로 그룹화하여 행 수 합계 및 파일 수 계산
summary_df = metadata_df.groupby(['키워드', '연도']).agg(
    총_파일_수=('파일명', 'count'),
    총_데이터_행_수=('총_행_수', 'sum')
).reset_index()

display(summary_df.sort_values(by=['키워드', '연도']))

print("\n전체 데이터셋 요약:")
print(f"전체 키워드: {metadata_df['키워드'].nunique()}개")
print(f"전체 연도: {metadata_df['연도'].nunique()}년 ({metadata_df['연도'].min()}~{metadata_df['연도'].max()})")
print(f"총 로드된 엑셀 파일 수: {metadata_df.shape[0]}개")
print(f"전체 데이터 행 수 합계: {metadata_df['총_행_수'].sum()}행")

메타데이터 추출 시작...

--- 메타데이터 정리 완료 ---
총 24개의 파일에 대한 메타데이터가 정리되었습니다.

메타데이터 DataFrame 미리보기:


,키워드,연도,파일명,총_행_수,총_열_수,컬럼_이름,언론사_별_개수
0,화제,2018,NewsResult_전국일간지_화제_20180101-201812...,2985,8,"[뉴스 식별자, 일자, 언론사, 제목, 통합 분류1, 통합 분류2, 통합 분류3, ...","{'서울신문': 724, '세계일보': 477, '국민일보': 393, '중앙일보'..."
1,화제,2019,NewsResult_전국일간지_화제_20190101-201912...,2919,8,"[뉴스 식별자, 일자, 언론사, 제목, 통합 분류1, 통합 분류2, 통합 분류3, ...","{'서울신문': 606, '세계일보': 442, '아시아투데이': 357, '국민일..."
2,화제,2020,NewsResult_전국일간지_화제_20200101-202012...,2854,8,"[뉴스 식별자, 일자, 언론사, 제목, 통합 분류1, 통합 분류2, 통합 분류3, ...","{'세계일보': 538, '서울신문': 425, '국민일보': 359, '중앙일보'..."
3,화제,2021,NewsResult_전국일간지_화제_20210101-202112...,2388,8,"[뉴스 식별자, 일자, 언론사, 제목, 통합 분류1, 통합 분류2, 통합 분류3, ...","{'중앙일보': 407, '서울신문': 350, '세계일보': 328, '한국일보'..."
4,화제,2022,NewsResult_전국일간지_화제_20220101-202212...,1844,8,"[뉴스 식별자, 일자, 언론사, 제목, 통합 분류1, 통합 분류2, 통합 분류3, ...","{'서울신문': 323, '세계일보': 292, '중앙일보': 245, '조선일보'..."



키워드별, 연도별 데이터 요약:


,키워드,연도,총_파일_수,총_데이터_행_수
0,논란,2018,1,16759
1,논란,2019,1,20012
2,논란,2020,1,20199
3,논란,2021,1,21821
4,논란,2022,1,16080
5,논란,2023,1,14915
6,논란,2024,1,14697
7,논란,2025,1,14611
8,이슈,2018,1,2442
9,이슈,2019,1,2593



전체 데이터셋 요약:
전체 키워드: 3개
전체 연도: 8년 (2018~2025)
총 로드된 엑셀 파일 수: 24개
전체 데이터 행 수 합계: 176775행


In [ ]:
# 업데이트된 메타데이터 DataFrame을 엑셀 파일로 저장
output_file_path = os.path.join(base_path, 'metadata_summary.xlsx')

try:
    metadata_df.to_excel(output_file_path, index=False)
    print(f"\n업데이트된 메타데이터가 '{output_file_path}'에 성공적으로 저장되었습니다.")
except Exception as e:
    print(f"\n업데이트된 메타데이터 엑셀 파일 저장 중 오류 발생: {e}")


업데이트된 메타데이터가 '/content/drive/MyDrive/26 국어정보학/기말과제/metadata_summary.xlsx'에 성공적으로 저장되었습니다.


### 4. 각 기사 키워드 포함 문장 실제 추출

In [9]:
# --- 웹 스크래핑 헬퍼 함수 정의 ---
def fetch_article_content(url, timeout=3):
    """
    뉴스 기사 URL에서 본문 내용을 추출하는 함수.
    속도 최적화 버전
    """

    headers = {
        'User-Agent': 'Mozilla/5.0'
    }

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=timeout
        )

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            'lxml'
        )

        content_tags = [
            soup.find('div', class_='article_body'),
            soup.find('div', id='articleBodyContents'),
            soup.find('div', id='article-content'),
            soup.find('div', class_='article-view'),
            soup.find('div', class_='news_cnt'),
            soup.find('article', class_='article_content'),
            soup.find('div', itemprop='articleBody')
        ]

        for tag in content_tags:

            if tag:

                for junk_tag in tag.find_all([
                    'script',
                    'style',
                    'span',
                    'figcaption',
                    'em',
                    'strong',
                    'a',
                    'b'
                ]):
                    junk_tag.extract()

                return tag.get_text(
                    separator='\n',
                    strip=True
                )

        return None

    except Exception:
        return None

### 4-1-1. '화제' 키워드, 2018년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2018

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2018년 기사 문장 추출 시작 ---


'화제' (2018) 기사 처리 중:   0%|          | 0/2985 [00:00<?, ?it/s]


--- '화제' 키워드, 2018년 기사 문장 추출 완료 ---
총 1445개의 '화제' 키워드 문장이 2018년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2018,중앙일보,https://www.joongang.co.kr/article/23251051,이영자는 유명 잡지 표지 모델과 수영복 자태 공개로 화제가 됐습니다.
1,화제,2018,중앙일보,https://www.joongang.co.kr/article/23251051,지금 커뮤니티에서 큰 화제가 되고 있는 이슈들입니다.
2,화제,2018,국민일보,http://news.kmib.co.kr/article/view.asp?arcid=...,SBS 예능 프로그램 ‘미운 우리 새끼’에서 거침없는 입담으로 화제가 된 가수 홍진...
3,화제,2018,세계일보,http://www.segye.com/content/html/2018/12/31/2...,'흥자매'로 화제를 모은 가수 홍진영(오른쪽)의 언니 홍선영(왼쪽)이 가족을 향한 ...
4,화제,2018,문화일보,http://www.munhwa.com/news/view.html?no=201812...,가장 화제가 된 인사는 진옥동(1961년생) 신임 신한은행장이다.



--- '화제' 키워드, 2018년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2985개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1391개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 1594개
실패율: 53.40%
'화제' 키워드, 2018년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2018.xlsx'에 성공적으로 저장되었습니다.


### 4-1-2. '화제' 키워드, 2019년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2019

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2019년 기사 문장 추출 시작 ---


'화제' (2019) 기사 처리 중:   0%|          | 0/2919 [00:00<?, ?it/s]


--- '화제' 키워드, 2019년 기사 문장 추출 완료 ---
총 1703개의 '화제' 키워드 문장이 2019년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2019,국민일보,http://news.kmib.co.kr/article/view.asp?arcid=...,삼풍백화점 붕괴사고가 화제가 되면서 배우 김상경이 당시 구조대로 활동한 사실도 다시...
1,화제,2019,세계일보,http://www.segye.com/content/html/2019/12/30/2...,최근 미국 정부는 화학물질 부문 동물실험을 2035년부터 원칙적으로 금지하기로 해 ...
2,화제,2019,중앙일보,https://www.joongang.co.kr/article/23669686,2019년을 떠나보내는 오늘 밤 ‘제야의 종’ 타종 행사에 화제의 EBS 캐릭터 ‘...
3,화제,2019,세계일보,http://www.segye.com/content/html/2019/12/30/2...,우선 올 연말 임원 인사에서는 젊고 능력 있는 여성 임원의 발탁이 화제가 됐다.
4,화제,2019,아시아투데이,http://www.asiatoday.co.kr/view.php?key=201912...,배우 한지민의 드레스 자태가 화제다.



--- '화제' 키워드, 2019년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2919개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1585개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 1334개
실패율: 45.70%
'화제' 키워드, 2019년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2019.xlsx'에 성공적으로 저장되었습니다.


### 4-1-3. '화제' 키워드, 2020년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2020

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2020년 기사 문장 추출 시작 ---


'화제' (2020) 기사 처리 중:   0%|          | 0/2854 [00:00<?, ?it/s]


--- '화제' 키워드, 2020년 기사 문장 추출 완료 ---
총 2035개의 '화제' 키워드 문장이 2020년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2020,국민일보,http://news.kmib.co.kr/article/view.asp?arcid=...,예매로 국립극단 홈페이지가 일시 마비될 정도로 화제를 모았던 연극 ‘화전가’도 예수...
1,화제,2020,세계일보,http://www.segye.com/content/html/2020/12/31/2...,특히 이 설전은 두 사람의 과거 인연과 대조를 이루면서 화제가 됐다.
2,화제,2020,중앙일보,https://www.joongang.co.kr/article/23959232,지금 커뮤니티에서 큰 화제가 되고 있는 이슈들입니다.
3,화제,2020,한국일보,https://hankookilbo.com/News/Read/A20201231142...,"한국 야구계의 살아 있는 전설 양준혁과 예비 신부 박현선은 19살 나이 차이로, 열..."
4,화제,2020,중앙일보,https://www.joongang.co.kr/article/23959086,"2014년 ‘한국사회, 4인 논객이 말한다’를 시작으로 매년 토론자들의 뜨거운 대결..."



--- '화제' 키워드, 2020년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2854개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1816개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 1038개
실패율: 36.37%
'화제' 키워드, 2020년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2020.xlsx'에 성공적으로 저장되었습니다.


### 4-1-4. '화제' 키워드, 2021년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2021

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2021년 기사 문장 추출 시작 ---


'화제' (2021) 기사 처리 중:   0%|          | 0/2388 [00:00<?, ?it/s]


--- '화제' 키워드, 2021년 기사 문장 추출 완료 ---
총 1720개의 '화제' 키워드 문장이 2021년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2021,경향신문,https://www.khan.co.kr/opinion/yeojeok/article...,3중 턱살이 사라진 김정은 북한 국무위원장의 얼굴과 목도 화제다.
1,화제,2021,한국일보,https://hankookilbo.com/News/Read/A20211231135...,대서양 상공을 여행하던 비행기 안에서 신종 코로나바이러스 감염증(코로나19) 감염 ...
2,화제,2021,한국일보,https://hankookilbo.com/News/Read/A20211231135...,"영상이 화제가 되자, 언론의 인터뷰 요청도 쏟아졌다."
3,화제,2021,문화일보,http://www.munhwa.com/news/view.html?no=202112...,31일 정년퇴직한 윤종기(60·사진) 전 공무원연금공단 재해보상실장이 최근 2년의 ...
4,화제,2021,국민일보,http://news.kmib.co.kr/article/view.asp?arcid=...,해당 영상이 온라인상에서 화제를 모으자 경찰이 다시 수사를 해보겠다는 입장을 전해왔...



--- '화제' 키워드, 2021년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2388개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1520개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 868개
실패율: 36.35%
'화제' 키워드, 2021년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2021.xlsx'에 성공적으로 저장되었습니다.


### 4-1-5. '화제' 키워드, 2022년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2022

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2022년 기사 문장 추출 시작 ---


'화제' (2022) 기사 처리 중:   0%|          | 0/1844 [00:00<?, ?it/s]


--- '화제' 키워드, 2022년 기사 문장 추출 완료 ---
총 1175개의 '화제' 키워드 문장이 2022년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2022,경향신문,https://www.khan.co.kr/opinion/column/article/...,이주호 교육부 장관이 최근 여러 인터뷰를 통해 자신이 “수능 폐지론자”라거나 “수능...
1,화제,2022,중앙일보,https://www.joongang.co.kr/article/25130252,구조 이후 화제가 됐던 ‘커피믹스’도 그중 하나였다.
2,화제,2022,경향신문,https://www.khan.co.kr/national/national-gener...,아프가니스탄 특별기여자 자녀들이 울산지역 초등학교에 입학하면서 일부 학부모들이 반발...
3,화제,2022,세계일보,http://www.segye.com/content/html/2022/12/30/2...,"미국에서 한 여성이 차를 탄 채로 낭떠러지 아래로 추락했지만, ‘나의 아이폰 찾기’..."
4,화제,2022,중앙일보,https://www.joongang.co.kr/article/25130179,온라인 게임 대회 ‘리그오브레전드(LoL) 2022 월드 챔피언십’ 우승자 김혁규 ...



--- '화제' 키워드, 2022년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 1844개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1098개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 746개
실패율: 40.46%
'화제' 키워드, 2022년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2022.xlsx'에 성공적으로 저장되었습니다.


### 4-1-6. '화제' 키워드, 2023년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2023

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2023년 기사 문장 추출 시작 ---


'화제' (2023) 기사 처리 중:   0%|          | 0/2097 [00:00<?, ?it/s]


--- '화제' 키워드, 2023년 기사 문장 추출 완료 ---
총 1332개의 '화제' 키워드 문장이 2023년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2023,국민일보,https://news.kmib.co.kr/article/view.asp?arcid...,유튜버 카라큘라가 배우 고(故) 이선균씨를 협박해 돈을 갈취한 협박범의 신상을 공개...
1,화제,2023,세계일보,http://www.segye.com/content/html/2023/12/30/2...,"누리꾼들은 그의 수상 소감에 대해 ""최근 연예대상은 대상 수상자보다 김구라 소감이 ..."
2,화제,2023,세계일보,http://www.segye.com/content/html/2023/12/30/2...,앞서 김구라는 지난 2019년 SBS 연예대상에서도 '사이다' 발언으로 화제가 된 ...
3,화제,2023,경향신문,https://www.khan.co.kr/economy/economy-general...,해마다 큰 화제를 불러일으킨 10명의 과학자를 선정하는 ‘네이처 10’을 꼽으면서다.
4,화제,2023,세계일보,http://www.segye.com/content/html/2023/12/29/2...,KBS Joy ‘무엇이든 물어보살’ 248회에서는 평범한 직장을 관두고 가상 자산 ...



--- '화제' 키워드, 2023년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2097개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1220개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 877개
실패율: 41.82%
'화제' 키워드, 2023년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2023.xlsx'에 성공적으로 저장되었습니다.


### 4-1-7. '화제' 키워드, 2024년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2024

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2024년 기사 문장 추출 시작 ---


'화제' (2024) 기사 처리 중:   0%|          | 0/2174 [00:00<?, ?it/s]


--- '화제' 키워드, 2024년 기사 문장 추출 완료 ---
총 1373개의 '화제' 키워드 문장이 2024년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2024,세계일보,http://www.segye.com/content/html/2024/12/30/2...,중국에서 출산을 앞둔 임산부가 진통을 참으면서 풀 메이크업을 하는 영상이 공개돼 화제다.
1,화제,2024,세계일보,http://www.segye.com/content/html/2024/12/29/2...,포스코노동조합은 기부금 1억 원 이외에도 노사상생기금 40억 원을 포항사랑상품권으로...
2,화제,2024,세계일보,http://www.segye.com/content/html/2024/12/27/2...,"지난해에는 6세 연하인 건축가 남자친구와 열애를 인정한 뒤, 방송을 통해 남자친구를..."
3,화제,2024,세계일보,http://www.segye.com/content/html/2024/12/27/2...,마약 상습 투약 혐의로 재판 중인 배우 유아인이 8급 매물로 63억원에 내놓은 이태...
4,화제,2024,세계일보,http://www.segye.com/content/html/2024/12/27/2...,특히 유아인이 MBC 예능 프로그램 ‘나 혼자 산다’에서 해당 주택을 공개해 화제가...



--- '화제' 키워드, 2024년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2174개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1261개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 913개
실패율: 42.00%
'화제' 키워드, 2024년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2024.xlsx'에 성공적으로 저장되었습니다.


### 4-1-8. '화제' 키워드, 2025년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '화제'
current_year = 2025

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜁니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5) # 서버 과부하 방지를 위한 딜레이를 0.1에서 0.5초로 증가

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '화제' 키워드, 2025년 기사 문장 추출 시작 ---


'화제' (2025) 기사 처리 중:   0%|          | 0/2161 [00:00<?, ?it/s]


--- '화제' 키워드, 2025년 기사 문장 추출 완료 ---
총 1302개의 '화제' 키워드 문장이 2025년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,화제,2025,서울신문,https://www.seoul.co.kr/news/newsView.php?id=2...,동명의 캐나다 드라마를 원작으로 한 이 작품은 삶과 죽음의 존엄성이라는 화두를 던지...
1,화제,2025,서울신문,https://www.seoul.co.kr/news/newsView.php?id=2...,MBC는 올 한 해 ‘시청률 가뭄’ 속에서도 비교적 화제성과 성과를 거둔 ‘언더커버...
2,화제,2025,문화일보,https://www.munhwa.com/article/11557672?ref=kpf,온라인에서 한 여성이 남편이 ‘여성 BJ’에게 거액을 후원했다는 사실을 알게 된 사...
3,화제,2025,세계일보,https://www.segye.com/newsView/20251229516529,이 와중에 최근 박혜수의 근황이 전해지며 화제가 되고 있다.
4,화제,2025,문화일보,https://www.munhwa.com/article/11557635?ref=kpf,미혼인 중년 여성들이 병원에서 우는 영상이 중국에서 화제가 되고 있다.



--- '화제' 키워드, 2025년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2161개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1175개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 986개
실패율: 45.63%
'화제' 키워드, 2025년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_화제_2025.xlsx'에 성공적으로 저장되었습니다.


### 4-2-1. '논란' 키워드, 2018년 기사에서 문장 추출 및 파일 저장

In [10]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2018

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2018년 기사 문장 추출 시작 ---


'논란' (2018) 기사 처리 중:   0%|          | 0/16759 [00:00<?, ?it/s]


--- '논란' 키워드, 2018년 기사 문장 추출 완료 ---
총 11236개의 '논란' 키워드 문장이 2018년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2018,내일신문,http://www.naeil.com/news_view/?id_art=299730,매출총량 배분 측면에서도 업종간 공정성 여부에 문제가 있다는 논란 등을 해소하기 위...
1,논란,2018,문화일보,http://www.munhwa.com/news/view.html?no=201812...,정권때마다 권력압력 논란
2,논란,2018,문화일보,http://www.munhwa.com/news/view.html?no=201812...,31일 재계에 따르면 공기업을 전신으로 하거나 주인 없는 민간 기업집단의 대표가 교...
3,논란,2018,중앙일보,https://www.joongang.co.kr/article/23250505,사단법인 대한정신장애인협회가 31일 경남도의회 브리핑룸에서 정신장애인 비하 발언 논...
4,논란,2018,중앙일보,https://www.joongang.co.kr/article/23250505,사단법인 대한정신장애인협회는 최근 이해찬 더불어민주당 대표의 정신장애인 비하 발언 ...



--- '논란' 키워드, 2018년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 16759개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 16759개
기사 내용 추출 성공 URL 개수: 8219개
기사 내용 추출 실패 URL 개수: 8540개
이번 실행에서의 실패율: 50.96%
현재까지 누적된 총 실패 URL 개수: 8540개
'논란' 키워드, 2018년 문장이 저장되었습니다.


### 4-2-2. '논란' 키워드, 2019년 기사에서 문장 추출 및 파일 저장

In [11]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2019

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2019년 기사 문장 추출 시작 ---


'논란' (2019) 기사 처리 중:   0%|          | 0/20010 [00:00<?, ?it/s]


--- '논란' 키워드, 2019년 기사 문장 추출 완료 ---
총 16005개의 '논란' 키워드 문장이 2019년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2019,중앙일보,https://www.joongang.co.kr/article/23510878,"그러나 실제로 황 대표의 아들이 명문대를 졸업하고 학점은 3.29, 토익은 925점..."
1,논란,2019,중앙일보,https://www.joongang.co.kr/article/23510878,황 대표의 아들이 취업한 기업이 채용 비리 문제가 크게 불거진 KT라는 점에서 논란...
2,논란,2019,중앙일보,https://www.joongang.co.kr/article/23511118,또 상산고 등에서 논란이 되는 사회통합전형 대상자 선발 지표도 정성평가로 이뤄져 불...
3,논란,2019,중앙일보,https://www.joongang.co.kr/article/23511118,민사고가 재지정 될 경우 상산고 평가의 불공정 논란은 더욱 거세질 것으로 보인다.
4,논란,2019,중앙일보,https://www.joongang.co.kr/article/23510852,논란이 된 질문과 답변이 지원자의 당락에는 영향을 미치지 않았다고 한다”라고 설명했다.



--- '논란' 키워드, 2019년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 20010개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 20010개
기사 내용 추출 성공 URL 개수: 11145개
기사 내용 추출 실패 URL 개수: 8865개
이번 실행에서의 실패율: 44.30%
현재까지 누적된 총 실패 URL 개수: 17405개
'논란' 키워드, 2019년 문장이 저장되었습니다.


### 4-2-3. '논란' 키워드, 2020년 기사에서 문장 추출 및 파일 저장

In [12]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2020

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2020년 기사 문장 추출 시작 ---


'논란' (2020) 기사 처리 중:   0%|          | 0/20197 [00:00<?, ?it/s]


--- '논란' 키워드, 2020년 기사 문장 추출 완료 ---
총 16827개의 '논란' 키워드 문장이 2020년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2020,한국일보,https://www.hankookilbo.com/News/Read/A2020063...,경기지역의 한 사립유치원이 회계부정이 드러난 6억여원을 무려 20년 간 나눠 갚는 ...
1,논란,2020,한국일보,https://www.hankookilbo.com/News/Read/A2020063...,그러나 하나같이 처벌 규정이 없어 실효성 논란이 컸다.
2,논란,2020,한국일보,https://www.hankookilbo.com/News/Read/A2020063...,매니저에 대한 '갑질' 논란에 휘말린 원로배우 이순재(85)가 이와 관련한 SBS ...
3,논란,2020,한국일보,https://www.hankookilbo.com/News/Read/A2020063...,논란이 커지자 이순재는 한발 물러서는 모습을 보였다.
4,논란,2020,한국일보,https://www.hankookilbo.com/News/Read/A2020063...,"한 고위 검사는 ""윤 총장이 일방적으로 자문단을 구성한다면 불공정 논란이 불가피하다..."



--- '논란' 키워드, 2020년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 20197개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 20197개
기사 내용 추출 성공 URL 개수: 11916개
기사 내용 추출 실패 URL 개수: 8281개
이번 실행에서의 실패율: 41.00%
현재까지 누적된 총 실패 URL 개수: 25686개
'논란' 키워드, 2020년 문장이 저장되었습니다.


### 4-2-4. '논란' 키워드, 2021년 기사에서 문장 추출 및 파일 저장

In [13]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2021

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2021년 기사 문장 추출 시작 ---


'논란' (2021) 기사 처리 중:   0%|          | 0/21819 [00:00<?, ?it/s]


--- '논란' 키워드, 2021년 기사 문장 추출 완료 ---
총 20002개의 '논란' 키워드 문장이 2021년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2021,한국일보,https://hankookilbo.com/News/Read/A20210630124...,검찰총장의 정치직행을 두고 중립성 논란이 적지 않지만 큰 걸림돌은 아니다.
1,논란,2021,한국일보,https://hankookilbo.com/News/Read/A20210630151...,논란된 '백운규 배임'은 수사심의위서 결론
2,논란,2021,한국일보,https://hankookilbo.com/News/Read/A20210630152...,'꼼수 이전' 논란으로 비어 있던 세종시 관세평가분류원 청사가 고용노동부 내 산업재...
3,논란,2021,서울신문,http://go.seoul.co.kr/news/newsView.php?id=202...,공교롭게도 감사 초점이 ‘지난해 한국수력원자력(한수원)이 공공기관 경영평가에서 A등...
4,논란,2021,서울신문,http://go.seoul.co.kr/news/newsView.php?id=202...,감사원이 민간위원들을 조사한 방식도 논란의 여지가 많다.



--- '논란' 키워드, 2021년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 21819개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 21819개
기사 내용 추출 성공 URL 개수: 13880개
기사 내용 추출 실패 URL 개수: 7939개
이번 실행에서의 실패율: 36.39%
현재까지 누적된 총 실패 URL 개수: 33625개
'논란' 키워드, 2021년 문장이 저장되었습니다.


### 4-2-5. '논란' 키워드, 2022년 기사에서 문장 추출 및 파일 저장

In [14]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2022

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2022년 기사 문장 추출 시작 ---


'논란' (2022) 기사 처리 중:   0%|          | 0/16078 [00:00<?, ?it/s]


--- '논란' 키워드, 2022년 기사 문장 추출 완료 ---
총 15271개의 '논란' 키워드 문장이 2022년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2022,한국일보,https://hankookilbo.com/News/Read/A20221230134...,김 처장은 '고발사주' 의혹 사건 등을 처리하면서 겪었던 통신자료 조회 논란에 대한...
1,논란,2022,중앙일보,https://www.joongang.co.kr/article/25130308,어린이집 선생님들이 근무시간에 23개월짜리 아이를 데리고 술집에서 생맥주와 치킨을 ...
2,논란,2022,한국일보,https://hankookilbo.com/News/Read/A20221230134...,'나는 솔로'의 잇따른 논란…부정적 의견 어쩌나
3,논란,2022,한국일보,https://hankookilbo.com/News/Read/A20221230134...,지난해 12월 방송된 SBS Plus·ENA PLAY '나는 솔로'에서 4기 영철...
4,논란,2022,중앙일보,https://www.joongang.co.kr/article/25130123,앞서 박 의원은 지난 2020년 11월 법사위에서 해당 예산이 작년 3000만원에서...



--- '논란' 키워드, 2022년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 16078개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 16078개
기사 내용 추출 성공 URL 개수: 10620개
기사 내용 추출 실패 URL 개수: 5458개
이번 실행에서의 실패율: 33.95%
현재까지 누적된 총 실패 URL 개수: 39083개
'논란' 키워드, 2022년 문장이 저장되었습니다.


### 4-2-6. '논란' 키워드, 2023년 기사에서 문장 추출 및 파일 저장

In [15]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2023

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2023년 기사 문장 추출 시작 ---


'논란' (2023) 기사 처리 중:   0%|          | 0/14914 [00:00<?, ?it/s]


--- '논란' 키워드, 2023년 기사 문장 추출 완료 ---
총 12859개의 '논란' 키워드 문장이 2023년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2023,한국일보,https://www.hankookilbo.com/News/Read/A2023123...,"논란 일기도, 수사 5개월 만에 혐의 벗어"
1,논란,2023,한국일보,https://www.hankookilbo.com/News/Read/A2023123...,수사과정에서 경찰이 공소시효가 지난 사체유기 혐의를 적용해 해당 여성을 유치장에 가...
2,논란,2023,한국일보,https://www.hankookilbo.com/News/Read/A2023123...,논란이 일자 경찰은 A씨를 체포 18시간여 만에 풀어줬다.
3,논란,2023,중앙일보,https://www.joongang.co.kr/article/25218725,교복 가격을 담합한 혐의로 벌금형을 받은 교복업자들이 가족 명의로 다시 입찰에 응해...
4,논란,2023,한국일보,https://www.hankookilbo.com/News/Read/A2023122...,"마스터 클래스는 처음엔 유명 연주자나 대학 출강 강사가 맡는 경우가 많았지만, 최근..."



--- '논란' 키워드, 2023년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 14914개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 14914개
기사 내용 추출 성공 URL 개수: 9833개
기사 내용 추출 실패 URL 개수: 5081개
이번 실행에서의 실패율: 34.07%
현재까지 누적된 총 실패 URL 개수: 44164개
'논란' 키워드, 2023년 문장이 저장되었습니다.


### 4-2-7. '논란' 키워드, 2024년 기사에서 문장 추출 및 파일 저장

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2024

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2024년 기사 문장 추출 시작 ---


'논란' (2024) 기사 처리 중:   0%|          | 0/14696 [00:00<?, ?it/s]


--- '논란' 키워드, 2024년 기사 문장 추출 완료 ---
총 13109개의 '논란' 키워드 문장이 2024년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2024,내일신문,https://www.naeil.com/news/read/533891?ref=big...,윤석열 대통령에 대한 체포영장이 발부되면서 수사권을 둘러싼 논란은 불식될 것으로 보인다.
1,논란,2024,한국일보,https://www.hankookilbo.com/News/Read/A2024123...,①직권남용죄는 대통령 불소추특권이 적용되는 범죄라서 어느 수사기관이든 수사 개시가 ...
2,논란,2024,한국일보,https://www.hankookilbo.com/News/Read/A2024123...,공수처는 법원의 영장 발부로 수사권 논란이 종결된 것이란 입장이다.
3,논란,2024,한국일보,https://www.hankookilbo.com/News/Read/A2024123...,윤석열 대통령에 대해 고위공직자범죄수사처(공수처)가 청구한 체포영장이 발부된 건 그...
4,논란,2024,세계일보,http://www.segye.com/content/html/2024/12/31/2...,"공수처는 이번 체포영장 발부로 수사의 적법성 논란도 일단락됐다고 보고 있지만, 향후..."



--- '논란' 키워드, 2024년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 14696개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 14696개
기사 내용 추출 성공 URL 개수: 9618개
기사 내용 추출 실패 URL 개수: 5078개
이번 실행에서의 실패율: 34.55%
현재까지 누적된 총 실패 URL 개수: 49242개
'논란' 키워드, 2024년 문장이 저장되었습니다.


### 4-2-8. '논란' 키워드, 2025년 기사에서 문장 추출 및 파일 저장

In [17]:
from concurrent.futures import ThreadPoolExecutor, as_completed

current_keyword = '논란'
current_year = 2025

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(
    current_keyword,
    current_year,
    all_excel_files
)

if df_keyword_year is None:
    print(
        f"경고: '{current_keyword}' 키워드에 대해 "
        f"{current_year}년도 파일을 찾을 수 없습니다."
    )

else:

    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []

    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = (
            df_keyword_year['URL']
            .dropna()
            .unique()
            .tolist()
        )

    initial_total_urls = len(all_unique_urls_in_excel)

    all_unique_urls_to_process = [
        url
        for url in all_unique_urls_in_excel
        if url not in globally_failed_urls
    ]

    skipped_urls_count = (
        initial_total_urls
        - len(all_unique_urls_to_process)
    )

    if skipped_urls_count > 0:
        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

    total_urls_to_process = len(
        all_unique_urls_to_process
    )

    if total_urls_to_process == 0:

        if initial_total_urls > 0:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일의 모든 URL이 "
                f"이전에 실패하여 건너뜁니다."
            )
        else:
            print(
                f"경고: '{current_keyword}' 키워드, "
                f"{current_year}년 파일에 URL이 없습니다."
            )

    else:

        # URL → 언론사 매핑
        url_to_media = (
            df_keyword_year
            .dropna(subset=['URL'])
            .drop_duplicates(subset=['URL'])
            .set_index('URL')['언론사']
            .to_dict()
        )

        def process_url(url):

            article_text = fetch_article_content(url)

            if not article_text:
                return {
                    "success": False,
                    "url": url
                }

            sentences = find_sentences_with_keyword(
                article_text,
                current_keyword
            )

            media = url_to_media.get(
                url,
                'N/A'
            )

            results = []

            for sentence in sentences:

                results.append({
                    '키워드': current_keyword,
                    '연도': current_year,
                    '언론사': media,
                    'URL': url,
                    '추출된_문장': sentence
                })

            return {
                "success": True,
                "url": url,
                "data": results
            }

        with ThreadPoolExecutor(
            max_workers=30
        ) as executor:

            futures = {
                executor.submit(
                    process_url,
                    url
                ): url
                for url in all_unique_urls_to_process
            }

            for future in tqdm(
                as_completed(futures),
                total=len(futures),
                desc=f"'{current_keyword}' ({current_year}) 기사 처리 중"
            ):

                try:

                    result = future.result()

                    if result["success"]:

                        extracted_sentences_data_current_block.extend(
                            result["data"]
                        )

                    else:

                        failed_urls_current_block.append(
                            result["url"]
                        )

                except Exception:

                    failed_urls_current_block.append(
                        futures[future]
                    )

        globally_failed_urls.update(
            failed_urls_current_block
        )

        current_keyword_year_df = pd.DataFrame(
            extracted_sentences_data_current_block
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 기사 문장 추출 완료 ---"
        )

        print(
            f"총 {len(current_keyword_year_df)}개의 "
            f"'{current_keyword}' 키워드 문장이 "
            f"{current_year}년 기사에서 추출되었습니다."
        )

        display(
            current_keyword_year_df.head()
        )

        successful_url_fetches = (
            total_urls_to_process
            - len(failed_urls_current_block)
        )

        failed_url_fetches = len(
            failed_urls_current_block
        )

        failed_percentage_of_processed = (
            failed_url_fetches
            / total_urls_to_process
            * 100
        )

        print(
            f"\n--- '{current_keyword}' 키워드, "
            f"{current_year}년 URL 처리 결과 요약 ---"
        )

        print(
            f"전체 엑셀 파일 내 고유 URL 개수: "
            f"{initial_total_urls}개"
        )

        print(
            f"이전에 실패하여 건너뛴 URL: "
            f"{skipped_urls_count}개"
        )

        print(
            f"이번 실행에서 처리 시도한 URL: "
            f"{total_urls_to_process}개"
        )

        print(
            f"기사 내용 추출 성공 URL 개수: "
            f"{successful_url_fetches}개"
        )

        print(
            f"기사 내용 추출 실패 URL 개수: "
            f"{failed_url_fetches}개"
        )

        print(
            f"이번 실행에서의 실패율: "
            f"{failed_percentage_of_processed:.2f}%"
        )

        print(
            f"현재까지 누적된 총 실패 URL 개수: "
            f"{len(globally_failed_urls)}개"
        )

        keyword_specific_extracted_data[
            current_keyword
        ].append(
            current_keyword_year_df
        )

        all_extracted_sentences_for_combined_save.append(
            current_keyword_year_df
        )

        output_keyword_year_path = os.path.join(
            base_path,
            f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx'
        )

        try:

            if not current_keyword_year_df.empty:

                current_keyword_year_df[
                    [
                        '언론사',
                        '추출된_문장',
                        '키워드',
                        '연도',
                        'URL'
                    ]
                ].to_excel(
                    output_keyword_year_path,
                    index=False
                )

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 문장이 "
                    f"저장되었습니다."
                )

            else:

                print(
                    f"'{current_keyword}' 키워드, "
                    f"{current_year}년 추출된 문장이 없습니다."
                )

        except Exception as e:

            print(
                f"엑셀 저장 오류: {e}"
            )


--- '논란' 키워드, 2025년 기사 문장 추출 시작 ---


'논란' (2025) 기사 처리 중:   0%|          | 0/14609 [00:00<?, ?it/s]


--- '논란' 키워드, 2025년 기사 문장 추출 완료 ---
총 12539개의 '논란' 키워드 문장이 2025년 기사에서 추출되었습니다.


,키워드,연도,언론사,URL,추출된_문장
0,논란,2025,한국일보,https://www.hankookilbo.com/News/Read/A2025123...,그는 청문회 내내 목소리를 높이거나 국회의 발언 중단 요구를 따르지 않아 논란이 됐다.
1,논란,2025,한국일보,https://www.hankookilbo.com/News/Read/A2025123...,A씨는 논란이 일자 취임 7개월 만인 2023년 4월 자진 사임했다.
2,논란,2025,한국일보,https://www.hankookilbo.com/News/Read/A2025123...,최근 불거진 연예인들의 미등록 기획사 논란 의식한 것으로 보여
3,논란,2025,한국일보,https://www.hankookilbo.com/News/Read/A2025123...,이 가운데 최근 송강호 최수종 송윤아 설경구 박나래 성시경 이하늬 바다 이지혜 강동...
4,논란,2025,한국일보,https://www.hankookilbo.com/News/Read/A2025123...,이 논란으로 인해 유아인은 현재 활동을 중단했다.



--- '논란' 키워드, 2025년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 14609개
이전에 실패하여 건너뛴 URL: 0개
이번 실행에서 처리 시도한 URL: 14609개
기사 내용 추출 성공 URL 개수: 8993개
기사 내용 추출 실패 URL 개수: 5616개
이번 실행에서의 실패율: 38.44%
현재까지 누적된 총 실패 URL 개수: 54858개
'논란' 키워드, 2025년 문장이 저장되었습니다.


### 4-3-1. '이슈' 키워드, 2018년 기사에서 문장 추출 및 파일 저장

In [18]:
current_keyword = '이슈'
current_year = 2018

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '이슈' 키워드, 2018년 기사 문장 추출 시작 ---


'이슈' (2018) 기사 처리 중:   0%|          | 0/2442 [00:00<?, ?it/s]


--- '이슈' 키워드, 2018년 기사 문장 추출 완료 ---
총 1230개의 '이슈' 키워드 문장이 2018년 기사에서 추출되었습니다.
추출된 문장 DataFrame 미리보기:


,키워드,연도,언론사,URL,추출된_문장
0,이슈,2018,중앙일보,https://www.joongang.co.kr/article/23251051,지금 커뮤니티에서 큰 화제가 되고 있는 이슈들입니다.
1,이슈,2018,중앙일보,https://www.joongang.co.kr/article/23252104,응급실 폭행 문제가 사회적 이슈로 부각되면서 최근 응급실 폭행 처벌을 강화하는 법률...
2,이슈,2018,세계일보,http://www.segye.com/content/html/2018/12/31/2...,남녀혐오 문제가 사회적 이슈로 부각하면서 페미니즘과 혜화역 시위 등 성별에 따라 입...
3,이슈,2018,세계일보,http://www.segye.com/content/html/2018/12/31/2...,"최근에는 성별에 따라 입장이 갈리기 쉬운 혜화역 시위, 이수역 폭행과 같은 젠더 이..."
4,이슈,2018,중앙일보,https://www.joongang.co.kr/article/23251322,"그는 ""학교폭력이라는 명목으로 글이 올라오고 있는 걸 알고 있었지만 그 글에 관심을..."



--- '이슈' 키워드, 2018년 URL 처리 결과 요약 ---
전체 엑셀 파일 내 고유 URL 개수: 2442개
기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): 1340개
기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): 1102개
실패율: 45.13%
'이슈' 키워드, 2018년 문장이 '/content/drive/MyDrive/26 국어정보학/기말과제/extracted_keyword_sentences_이슈_2018.xlsx'에 성공적으로 저장되었습니다.


### 4-3-2. '이슈' 키워드, 2019년 기사에서 문장 추출 및 파일 저장

In [19]:
current_keyword = '이슈'
current_year = 2019

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")


--- '이슈' 키워드, 2019년 기사 문장 추출 시작 ---


'이슈' (2019) 기사 처리 중:   0%|          | 0/2593 [00:00<?, ?it/s]

KeyboardInterrupt: 

### 4-3-3. '이슈' 키워드, 2020년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2020

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-4. '이슈' 키워드, 2021년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2021

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-5. '이슈' 키워드, 2022년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2022

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-6. '이슈' 키워드, 2023년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2023

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-7. '이슈' 키워드, 2024년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2024

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")

### 4-3-8. '이슈' 키워드, 2025년 기사에서 문장 추출 및 파일 저장

In [ ]:
current_keyword = '이슈'
current_year = 2025

print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 시작 ---")

df_keyword_year, filename = get_df_for_keyword_year(current_keyword, current_year, all_excel_files)

if df_keyword_year is None:
    print(f"경고: '{current_keyword}' 키워드에 대해 {current_year}년도 파일을 찾을 수 없습니다.")
else:
    extracted_sentences_data_current_block = []
    failed_urls_current_block = []

    all_unique_urls_in_excel = []
    if 'URL' in df_keyword_year.columns:
        all_unique_urls_in_excel = df_keyword_year['URL'].dropna().unique().tolist()

    total_urls_in_excel = len(all_unique_urls_in_excel)

    if total_urls_in_excel == 0:
        print(f"경고: '{current_keyword}' 키워드, {current_year}년 파일에 'URL' 컬럼이 없거나 모든 URL이 비어있습니다. 건너뜨니다.")
    else:
        with tqdm(total=total_urls_in_excel, desc=f"'{current_keyword}' ({current_year}) 기사 처리 중") as pbar:
            for url in all_unique_urls_in_excel:
                import io
                import contextlib
                with contextlib.redirect_stdout(io.StringIO()):
                    article_text = fetch_article_content(url)

                if article_text:
                    sentences = find_sentences_with_keyword(article_text, current_keyword)
                    # '언론사'는 해당 URL이 원본 데이터에 여러 번 나타날 경우 불일치할 수 있으므로, 해당 URL과 일치하는 첫 번째 언론사를 사용합니다.
                    media = 'N/A'
                    matching_rows = df_keyword_year[df_keyword_year['URL'] == url]
                    if not matching_rows.empty and '언론사' in matching_rows.columns:
                        first_media = matching_rows['언론사'].dropna().iloc[0] if not matching_rows['언론사'].dropna().empty else 'N/A'
                        media = first_media

                    if sentences:
                        for sentence in sentences:
                            extracted_sentences_data_current_block.append({
                                '키워드': current_keyword,
                                '연도': current_year,
                                '언론사': media,
                                'URL': url,
                                '추출된_문장': sentence
                            })
                else:
                    failed_urls_current_block.append(url) # Store failed URL
                pbar.update(1)
                time.sleep(0.5)

        current_keyword_year_df = pd.DataFrame(extracted_sentences_data_current_block)

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 문장 추출 완료 ---")
        print(f"총 {len(current_keyword_year_df)}개의 '{current_keyword}' 키워드 문장이 {current_year}년 기사에서 추출되었습니다.")
        print("추출된 문장 DataFrame 미리보기:")
        display(current_keyword_year_df.head())

        # URL 처리 결과 요약
        successful_url_fetches = total_urls_in_excel - len(failed_urls_current_block)
        failed_url_fetches = len(failed_urls_current_block)
        failed_percentage = (failed_url_fetches / total_urls_in_excel * 100) if total_urls_in_excel > 0 else 0

        print(f"\n--- '{current_keyword}' 키워드, {current_year}년 URL 처리 결과 요약 ---")
        print(f"전체 엑셀 파일 내 고유 URL 개수: {total_urls_in_excel}개")
        print(f"기사 내용 추출 성공 URL 개수 (텍스트 추출 성공): {successful_url_fetches}개")
        print(f"기사 내용 추출 실패 URL 개수 (텍스트 추출 실패): {failed_url_fetches}개")
        print(f"실패율: {failed_percentage:.2f}%")


        # if failed_urls_current_block:
        #     print(f"\n--- '{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL 목록 ({len(failed_urls_current_block)}개) ---")
        #     for failed_url in failed_urls_current_block:
        #         print(failed_url)
        # else:
        #     print(f"\n'{current_keyword}' 키워드, {current_year}년 기사 중 내용 추출에 실패한 URL은 없습니다.")

        # 키워드별 추출된 데이터 리스트에 추가 (나중에 키워드별 통합 파일을 위해)
        keyword_specific_extracted_data[current_keyword].append(current_keyword_year_df)
        # 모든 키워드의 통합 저장을 위한 리스트에 추가
        all_extracted_sentences_for_combined_save.append(current_keyword_year_df)

        # 현재 키워드-연도 데이터를 별도 엑셀 파일로 저장
        output_keyword_year_path = os.path.join(base_path, f'extracted_keyword_sentences_{current_keyword}_{current_year}.xlsx')
        try:
            if not current_keyword_year_df.empty:
                current_keyword_year_df[['언론사', '추출된_문장', '키워드', '연도', 'URL']].to_excel(output_keyword_year_path, index=False)
                print(f"'{current_keyword}' 키워드, {current_year}년 문장이 '{output_keyword_year_path}'에 성공적으로 저장되었습니다.")
            else:
                print(f"'{current_keyword}' 키워드, {current_year}년 추출된 문장이 없어 파일을 저장하지 않습니다.")
        except Exception as e:
            print(f"'{current_keyword}' 키워드, {current_year}년 엑셀 파일 저장 중 오류 발생: {e}")